The implementation of **MADDPG** algorithm.

Article Name: **Multi-Agent Actor-Critic for Mixed Cooperative-Competitive Environments**

The link: https://proceedings.neurips.cc/paper_files/paper/2017/hash/68a9750337a418a86fe06c1991a1d64c-Abstract.html

In [ ]:
!pip install mpe2

In [ ]:
from mpe2 import simple_adversary_v3

env = simple_adversary_v3.env(max_cycles=25, continuous_actions=True)
env.reset()

print("List of agents:", env.agents)
print("Action space for first agent:", env.action_space(env.agents[0]))
print("Observation space for first agent:", env.observation_space(env.agents[0]))

List of agents: ['adversary_0', 'agent_0', 'agent_1']
Action space for first agent: Box(0.0, 1.0, (5,), float32)
Observation space for first agent: Box(-inf, inf, (8,), float32)


In [ ]:
import os
import numpy as np
import torch as T
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from mpe2 import simple_adversary_v3

In [ ]:
logging.getLogger("pettingzoo").setLevel(logging.ERROR)


class MultiAgentReplayBuffer:
    def __init__(self, max_size, critic_dims, actor_dims, n_actions, n_agents, batch_size):
        self.mem_size = max_size
        self.mem_cntr = 0
        self.n_agents = n_agents
        self.actor_dims = actor_dims
        self.batch_size = batch_size
        self.n_actions = n_actions

        self.state_memory = np.zeros((self.mem_size, critic_dims))
        self.new_state_memory = np.zeros((self.mem_size, critic_dims))
        self.reward_memory = np.zeros((self.mem_size, n_agents))
        self.terminal_memory = np.zeros((self.mem_size, n_agents), dtype=bool)

        self.init_actor_memory()

    def init_actor_memory(self):
        self.actor_state_memory = []
        self.actor_new_state_memory = []
        self.actor_action_memory = []

        for i in range(self.n_agents):
            self.actor_state_memory.append(np.zeros((self.mem_size, self.actor_dims[i])))
            self.actor_new_state_memory.append(np.zeros((self.mem_size, self.actor_dims[i])))
            self.actor_action_memory.append(np.zeros((self.mem_size, self.n_actions)))

    def store_transition(self, raw_obs, state, action, reward, raw_obs_, state_, done):
        index = self.mem_cntr % self.mem_size

        for agent_idx in range(self.n_agents):
            self.actor_state_memory[agent_idx][index] = raw_obs[agent_idx]
            self.actor_new_state_memory[agent_idx][index] = raw_obs_[agent_idx]
            self.actor_action_memory[agent_idx][index] = action[agent_idx]

        self.state_memory[index] = state
        self.new_state_memory[index] = state_
        self.reward_memory[index] = reward
        self.terminal_memory[index] = done

        self.mem_cntr += 1

    def sample_buffer(self):
        max_mem = min(self.mem_cntr, self.mem_size)
        batch = np.random.choice(max_mem, self.batch_size, replace=False)

        states = self.state_memory[batch]
        states_ = self.new_state_memory[batch]
        rewards = self.reward_memory[batch]
        terminal = self.terminal_memory[batch]

        actor_states = []
        actor_new_states = []
        actions = []

        for agent_idx in range(self.n_agents):
            actor_states.append(self.actor_state_memory[agent_idx][batch])
            actor_new_states.append(self.actor_new_state_memory[agent_idx][batch])
            actions.append(self.actor_action_memory[agent_idx][batch])

        return actor_states, states, actions, rewards, actor_new_states, states_, terminal

    def ready(self):
        return self.mem_cntr >= self.batch_size


class CriticNetwork(nn.Module):
    def __init__(self, beta, input_dims, fc1_dims, fc2_dims, n_agents, n_actions, name, chkpt_dir='tmp/maddpg'):
        super(CriticNetwork, self).__init__()
        self.chkpt_file = os.path.join(chkpt_dir, name)

        self.fc1 = nn.Linear(input_dims + n_agents * n_actions, fc1_dims)
        self.fc2 = nn.Linear(fc1_dims, fc2_dims)
        self.q = nn.Linear(fc2_dims, 1)

        self.optimizer = optim.Adam(self.parameters(), lr=beta)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        self.to(self.device)

    def forward(self, state, action):
        x = F.relu(self.fc1(T.cat([state, action], dim=1)))
        x = F.relu(self.fc2(x))
        return self.q(x)

    def save_checkpoint(self):
        os.makedirs(os.path.dirname(self.chkpt_file), exist_ok=True)
        T.save(self.state_dict(), self.chkpt_file)

    def load_checkpoint(self):
        self.load_state_dict(T.load(self.chkpt_file))


class ActorNetwork(nn.Module):
    def __init__(self, alpha, input_dims, fc1_dims, fc2_dims, n_actions, name, chkpt_dir='tmp/maddpg'):
        super(ActorNetwork, self).__init__()
        self.chkpt_file = os.path.join(chkpt_dir, name)

        self.fc1 = nn.Linear(input_dims, fc1_dims)
        self.fc2 = nn.Linear(fc1_dims, fc2_dims)
        self.pi = nn.Linear(fc2_dims, n_actions)

        self.optimizer = optim.Adam(self.parameters(), lr=alpha)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        self.to(self.device)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return T.sigmoid(self.pi(x))

    def save_checkpoint(self):
        os.makedirs(os.path.dirname(self.chkpt_file), exist_ok=True)
        T.save(self.state_dict(), self.chkpt_file)

    def load_checkpoint(self):
        self.load_state_dict(T.load(self.chkpt_file))


class Agent:
    def __init__(self, actor_dims, critic_dims, n_actions, n_agents, agent_idx, chkpt_dir='tmp/maddpg', alpha=0.01, beta=0.01, fc1=64, fc2=64, gamma=0.95, tau=0.01):
        self.gamma = gamma
        self.tau = tau
        self.n_actions = n_actions
        self.agent_name = f'agent_{agent_idx}'

        self.actor = ActorNetwork(alpha, actor_dims, fc1, fc2, n_actions, name=self.agent_name+'_actor', chkpt_dir=chkpt_dir)
        self.critic = CriticNetwork(beta, critic_dims, fc1, fc2, n_agents, n_actions, name=self.agent_name+'_critic', chkpt_dir=chkpt_dir)
        self.target_actor = ActorNetwork(alpha, actor_dims, fc1, fc2, n_actions, name=self.agent_name+'_target_actor', chkpt_dir=chkpt_dir)
        self.target_critic = CriticNetwork(beta, critic_dims, fc1, fc2, n_agents, n_actions, name=self.agent_name+'_target_critic', chkpt_dir=chkpt_dir)

        self.update_network_parameters(tau=1)

    def update_network_parameters(self, tau=None):
        if tau is None:
            tau = self.tau

        for target_param, param in zip(self.target_actor.parameters(), self.actor.parameters()):
            target_param.data.copy_(tau * param.data + (1.0 - tau) * target_param.data)

        for target_param, param in zip(self.target_critic.parameters(), self.critic.parameters()):
            target_param.data.copy_(tau * param.data + (1.0 - tau) * target_param.data)

    def choose_action(self, observation):
        state = T.tensor(np.array([observation]), dtype=T.float).to(self.actor.device)
        actions = self.actor.forward(state)
        noise = T.randn(self.n_actions).to(self.actor.device) * 0.1
        action = actions + noise
        action = T.clamp(action, 0.0, 1.0)
        return action.detach().cpu().numpy()[0]


class MADDPG:
    def __init__(self, actor_dims, critic_dims, n_agents, n_actions, scenario='simple_adversary_v3', alpha=0.01, beta=0.01, fc1=64, fc2=64, gamma=0.99, tau=0.01, chkpt_dir='tmp/maddpg/'):
        self.agents = []
        self.n_agents = n_agents
        self.n_actions = n_actions
        chkpt_dir = os.path.join(chkpt_dir, scenario)

        for agent_idx in range(self.n_agents):
            self.agents.append(Agent(actor_dims[agent_idx], critic_dims, n_actions, n_agents, agent_idx, alpha=alpha, beta=beta, fc1=fc1, fc2=fc2, gamma=gamma, tau=tau, chkpt_dir=chkpt_dir))

    def choose_action(self, raw_obs):
        actions = []
        for agent_idx, agent in enumerate(self.agents):
            action = agent.choose_action(raw_obs[agent_idx])
            actions.append(action)
        return actions

    def learn(self, memory):
        if not memory.ready():
            return

        actor_states, states, actions, rewards, actor_new_states, states_, dones = memory.sample_buffer()
        device = self.agents[0].actor.device

        states = T.tensor(states, dtype=T.float).to(device)
        states_ = T.tensor(states_, dtype=T.float).to(device)
        rewards = T.tensor(rewards, dtype=T.float).to(device)
        dones = T.tensor(dones, dtype=T.bool).to(device)

        all_agents_new_actions = []
        old_agents_actions = []

        for agent_idx, agent in enumerate(self.agents):
            new_states = T.tensor(actor_new_states[agent_idx], dtype=T.float).to(device)
            new_pi = agent.target_actor.forward(new_states)
            all_agents_new_actions.append(new_pi)

            agent_actions = T.tensor(actions[agent_idx], dtype=T.float).to(device)
            old_agents_actions.append(agent_actions)

        new_actions = T.cat(all_agents_new_actions, dim=1)
        old_actions = T.cat(old_agents_actions, dim=1)

        for agent_idx, agent in enumerate(self.agents):
            with T.no_grad():
                critic_value_ = agent.target_critic.forward(states_, new_actions).flatten()
                critic_value_[dones[:, agent_idx]] = 0.0
                target = rewards[:, agent_idx] + agent.gamma * critic_value_

            critic_value = agent.critic.forward(states, old_actions).flatten()
            critic_loss = F.mse_loss(target, critic_value)

            agent.critic.optimizer.zero_grad()
            critic_loss.backward()
            agent.critic.optimizer.step()

        for agent_idx, agent in enumerate(self.agents):
            mu_actions = []
            for idx, a in enumerate(self.agents):
                current_states = T.tensor(actor_states[idx], dtype=T.float).to(device)
                if idx == agent_idx:
                    pi = a.actor.forward(current_states)
                else:
                    with T.no_grad():
                        pi = a.actor.forward(current_states)
                mu_actions.append(pi)

            mu_actions = T.cat(mu_actions, dim=1)
            actor_loss = -agent.critic.forward(states, mu_actions).mean()

            agent.actor.optimizer.zero_grad()
            actor_loss.backward()
            agent.actor.optimizer.step()

            agent.update_network_parameters()


def obs_list_to_state_vector(observation):
    return np.concatenate([obs for obs in observation])

In [ ]:
scenario = "simple_adversary_v3"
env = simple_adversary_v3.parallel_env(max_cycles=25, continuous_actions=True)
observations, infos = env.reset()

agents_list = env.agents
n_agents = len(agents_list)
actor_dims = [env.observation_space(agent).shape[0] for agent in agents_list]
critic_dims = sum(actor_dims)
n_actions = env.action_space(agents_list[0]).shape[0]

maddpg_agents = MADDPG(actor_dims, critic_dims, n_agents, n_actions, scenario=scenario)
memory = MultiAgentReplayBuffer(100000, critic_dims, actor_dims, n_actions, n_agents, batch_size=1024)

PRINT_INTERVAL = 100
N_GAMES = 1000
total_steps = 0
score_history = []

for i in range(N_GAMES):
    obs_dict, infos = env.reset()
    obs = [obs_dict[agent] for agent in agents_list]
    score = 0
    done = False

    while env.agents:
        actions_list = maddpg_agents.choose_action(obs)
        actions_dict = {agent: actions_list[idx] for idx, agent in enumerate(agents_list)}

        obs_dict_, rewards_dict, terminations, truncations, infos = env.step(actions_dict)

        obs_ = [obs_dict_[agent] for agent in agents_list if agent in obs_dict_]
        rewards = [rewards_dict[agent] for agent in agents_list if agent in rewards_dict]
        dones = [terminations[agent] or truncations[agent] for agent in agents_list if agent in terminations]

        if len(obs_) < n_agents:
            break

        state = obs_list_to_state_vector(obs)
        state_ = obs_list_to_state_vector(obs_)

        memory.store_transition(obs, state, actions_list, rewards, obs_, state_, dones)

        if total_steps % 100 == 0:
            maddpg_agents.learn(memory)

        obs = obs_
        score += sum(rewards)
        total_steps += 1

    score_history.append(score)
    avg_score = np.mean(score_history[-100:])

    if i % PRINT_INTERVAL == 0 and i > 0:
        print(f'episode {i} | average score: {avg_score:.1f}')

episode 100 | average score: -18.4
episode 200 | average score: -37.4
episode 300 | average score: -37.4
episode 400 | average score: -42.5
episode 500 | average score: -40.6
episode 600 | average score: -35.2
episode 700 | average score: -41.7
episode 800 | average score: -35.4
episode 900 | average score: -36.2
